# 1 Start (python 3.10.11)

In [1]:
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from bs4 import BeautifulSoup

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

pd.set_option('display.max_colwidth', 200)
headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_5) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.102 Safari/537.36'}
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

pd.set_option('display.max_columns', None)


luoghi=gpd.read_file("sn_json/luoghi.json")
curiosita=gpd.read_file('sn_json/curiosita.json')
personaggi=gpd.read_file('sn_json/personaggi.json')
cucina=gpd.read_file('sn_json/cucina.json')
leggende=gpd.read_file('sn_json/leggende.json')
eccellenze=gpd.read_file('sn_json/eccellenze.json')
interviste=gpd.read_file('sn_json/interviste.json')



def get_all(url):
    # Effettua la richiesta HTTP per ottenere il contenuto della pagina
    response = requests.get(url,headers=headers,verify=False)
    html_content = response.text

    # Parsing dell'HTML con BeautifulSoup
    soup = BeautifulSoup(html_content, 'html.parser')

    # Trova il tag meta con attributo property="og:title"
    og_title_tag = soup.find('meta', attrs={'property': 'og:title'})
    title = og_title_tag['content']

    # Trova il tag meta con attributo property="twitter:data1"
    og_author_tag = soup.find('meta', attrs={'name': 'twitter:data1'})
        # Estrai author
    if og_author_tag and 'content' in og_author_tag.attrs:
        author = og_author_tag['content']
        
    else:
        author=''

    # Trova il tag meta con attributo property="og:description"
    og_description_tag = soup.find('meta', attrs={'property': 'og:description'})
        # Estrai l'URL dell'immagine
    if og_description_tag and 'content' in og_description_tag.attrs:
        description = og_description_tag['content']
        
    else:
        description=''



    # Trova il tag meta con attributo property="og:image"
    og_image_tag = soup.find('meta', attrs={'property': 'og:image'})

    # Estrai l'URL dell'immagine
    if og_image_tag and 'content' in og_image_tag.attrs:
        image_url = og_image_tag['content']
        
    else:
        image_url=''
    infos={'title':title,'author':author,'description':description,'image_url':image_url}
    #print(infos['description'])
    return infos


def get_description(url):
    # Effettua la richiesta HTTP per ottenere il contenuto della pagina
    response = requests.get(url,headers=headers,verify=False)
    html_content = response.text

    # Parsing dell'HTML con BeautifulSoup
    soup = BeautifulSoup(html_content, 'html.parser')

    

    # Trova il tag meta con attributo property="og:description"
    og_description_tag = soup.find('meta', attrs={'property': 'og:description'})
        # Estrai l'URL dell'immagine
    if og_description_tag and 'content' in og_description_tag.attrs:
        description = og_description_tag['content']
        
    else:
        description=''

    #print(infos['description'])
    return description


def train_sn_model(df,titolo,descr):
    # Suddivisione dei dati in training set e test set
    X = df['titolo'] + ' ' + df['descr']  # Concateniamo titolo e descrizione
    y = df['category']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


    # Estrai le feature dai testi utilizzando la rappresentazione TF-IDF (Term Frequency-Inverse Document Frequency)
    vectorizer = TfidfVectorizer()
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)

    # Addestra il modello SVM
    svm = SVC(kernel='linear')  # Puoi sperimentare con diversi kernel
    svm.fit(X_train_tfidf, y_train)

        # Effettua le previsioni
    y_pred = svm.predict(X_test_tfidf)
    new_data = [titolo,descr]
    new_data_tfidf = vectorizer.transform(new_data)
    predicted_label = svm.predict(new_data_tfidf)
    return predicted_label[0]


def insert_story(df,url,category,lon,lat):
    infos=get_all(url)

    df.loc[len(df),['type','geometry','titolo','img','author','descr','url','category']] = ['Feature',Point(lon, lat),infos['title'],infos['image_url'],infos['author'],infos['description'],url,category]
  
def get_tags(tags):
    tags_name=[]
    for tag in tags:
        name = requests.get("https://storienapoli.it/wp-json/wp/v2/tags/"+str(tag),verify=False)
        tags_name.append(name.json()['name'])
    return tags_name



posts = requests.get("https://storienapoli.it/wp-json/wp/v2/posts",verify=False)
posts_df = pd.DataFrame(posts.json())
for index,row in posts_df.iterrows():
    description=get_description(row['link'])

    category = train_sn_model(pd.concat([luoghi,curiosita,personaggi,cucina,leggende,eccellenze,interviste]),row['title']['rendered'],description)
    if category=='luoghi':
        print("insert_story(luoghi,'"+row['link']+"','"+category+"', lon , lat)")
    if category=='curiosita':
        print("insert_story(curiosita,'"+row['link']+"','"+category+"', lon , lat)")
    if category=='personaggi':
        print("insert_story(personaggi,'"+row['link']+"','"+category+"', lon , lat)")
    if category=='leggende':
        print("insert_story(leggende,'"+row['link']+"','"+category+"', lon , lat)")
    if category=='cucina':
        print("insert_story(cucina,'"+row['link']+"','"+category+"', lon , lat)")
    if category=='interviste':
        print("insert_story(interviste,'"+row['link']+"','"+category+"', lon , lat)")
    if category=='eccellenze':
        print("insert_story(eccellenze,'"+row['link']+"','"+category+"', lon , lat)")


import leafmap.foliumap as leafmap
import folium
# Crea una mappa
m = leafmap.Map(location=[41,14], zoom_start=9)  # Imposta la posizione e lo zoom iniziali
m.add_child(folium.LatLngPopup())
m

insert_story(luoghi,'https://storienapoli.it/2025/05/30/scomparsi-senza-lasciar-traccia-il-mistero-del-monte-maggiore/','luoghi', lon , lat)
insert_story(personaggi,'https://storienapoli.it/2025/05/26/le-migliori-attrazioni-da-scoprire-a-napoli-dopo-il-tramonto/','personaggi', lon , lat)
insert_story(personaggi,'https://storienapoli.it/2025/05/23/gennaro-pandolfi-ed-il-piccolo-nunzio-la-camorra-uccide-il-futuro/','personaggi', lon , lat)
insert_story(personaggi,'https://storienapoli.it/2025/05/22/la-broda-di-castagne/','personaggi', lon , lat)
insert_story(luoghi,'https://storienapoli.it/2025/05/21/luoghi-storici-di-napoli-presentati-in-una-serie-di-slot-machine/','luoghi', lon , lat)
insert_story(luoghi,'https://storienapoli.it/2025/05/18/il-conclave-nella-citta-di-napoli/','luoghi', lon , lat)
insert_story(luoghi,'https://storienapoli.it/2025/05/16/la-strage-del-venerdi-santo/','luoghi', lon , lat)
insert_story(luoghi,'https://storienapoli.it/2025/05/15/le-origini-segrete-del-blackja

In [2]:
insert_story(curiosita,'https://storienapoli.it/2025/05/30/scomparsi-senza-lasciar-traccia-il-mistero-del-monte-maggiore/','curiosita', 14.1969 , 41.2372)
insert_story(personaggi,'https://storienapoli.it/2025/05/23/gennaro-pandolfi-ed-il-piccolo-nunzio-la-camorra-uccide-il-futuro/','personaggi', 14.2455 , 40.8616)
insert_story(cucina,'https://storienapoli.it/2025/05/22/la-broda-di-castagne/','cucina', 15.0181 , 40.8424)
insert_story(luoghi,'https://storienapoli.it/2025/05/18/il-conclave-nella-citta-di-napoli/','luoghi', 14.2595 , 40.8525)
insert_story(curiosita,'https://storienapoli.it/2025/05/14/le-velate-tra-sammartino-e-jago/','curiosita', 14.2515 , 40.8608)


## Get All , input url in ciclo for , cambiare url vecchio con quello nuovo 

Create geoDataframe (workaround to gpd.read_file() )

In [ ]:
def create_gdf(feature_name):
    df = requests.get("https://raw.githubusercontent.com/deinic/sn_json/main/"+feature_name+".json",verify=False)
    df = pd.DataFrame(df.json()['features'])
    df['geometry'] = df['geometry'].apply(lambda row: Point(row['coordinates'][0], row['coordinates'][1]))

    df['titolo']=None
    df['img']=None
    df['author']=None
    df['descr']=None
    df['url']=None
    df['category']=None

    for index,row in df.iterrows():
        df['titolo'].iloc[index] = df['properties'].iloc[index]['titolo']
        df['img'].iloc[index] = df['properties'].iloc[index]['img']
        df['author'].iloc[index] = df['properties'].iloc[index]['author']
        df['descr'].iloc[index] = df['properties'].iloc[index]['descr']
        df['url'].iloc[index] = df['properties'].iloc[index]['url']
        df['category'].iloc[index] = feature_name

    df = df.drop(columns=['properties'])
    gdf=gpd.GeoDataFrame(df,geometry=df['geometry'])
    return gdf

luoghi=create_gdf('luoghi')
curiosita=create_gdf('curiosita')
personaggi=create_gdf('personaggi')
cucina=create_gdf('cucina')
leggende=create_gdf('leggende')
eccellenze=create_gdf('eccellenze')
interviste=create_gdf('interviste')



# 2 Or recall from internal repo

In [3]:
luoghi=gpd.read_file("sn_json/luoghi.json")
curiosita=gpd.read_file('sn_json/curiosita.json')
personaggi=gpd.read_file('sn_json/personaggi.json')
cucina=gpd.read_file('sn_json/cucina.json')
leggende=gpd.read_file('sn_json/leggende.json')
eccellenze=gpd.read_file('sn_json/eccellenze.json')
interviste=gpd.read_file('sn_json/interviste.json')

# Check  and Change URL e IMG

In [6]:
def check_url(df,urls):
    count=0
    new_nok = pd.DataFrame()
    for url in urls:
        response=requests.get(url, headers=headers,verify=False)
        if ((response.url=='https://storienapoli.it') | (response.status_code != 200)):
            #print('URL NOT OK: '+ url)
            #display(df[df.url==url])
            not_ok=df[df.url==url]
            new_nok = pd.concat([new_nok,not_ok])
            count=count+1
    if count==0:
        print('Nessun Url corrotto')
    return new_nok
    
def change_url(df,df_nok_url):

    for index,row in df_nok_url.iterrows():
        search = requests.get("https://storienapoli.it//wp-json/wp/v2/search/?search="+row['titolo'])
        if len(search.json()) !=0 and row['titolo']==search.json()[0]['title']:
            df.loc[index, 'url'] = search.json()[0]['url']
        else:
            print(index)
            df.drop(index=index, inplace=True)

def get_ogr_image(url):
    # Effettua la richiesta HTTP per ottenere il contenuto della pagina
    response = requests.get(url,headers=headers,verify=False)
    html_content = response.text

    # Parsing dell'HTML con BeautifulSoup
    soup = BeautifulSoup(html_content, 'html.parser')

    # Trova il tag meta con attributo property="og:image"
    og_image_tag = soup.find('meta', attrs={'property': 'og:image'})

    # Estrai l'URL dell'immagine
    if og_image_tag and 'content' in og_image_tag.attrs:
        image_url = og_image_tag['content']
        
    else:
        image_url=''
    return image_url

def change_img(df,selection):
    count=0
    for i,row in selection.iterrows():
        img=row['img']
        url=row['url']
        if img == '' or requests.get(img, headers=headers,verify=False).status_code != 200:
            image_url=get_ogr_image(url)
            df.loc[df['img'] == img, 'img'] = image_url        
            count=count+1
            print("IMG cambiate")
    if count==0:
        print('Nessuna Img corrotta')

def check_img(df):
    count=0
    selection = gpd.GeoDataFrame()
    for img in df.img:
        if(img==''):
            selection=df[df['img']==img]
            display(selection[['titolo', 'img']])
            count=count+1
        elif (requests.get(img, headers=headers,verify=False).status_code != 200):
            selection=df[df['img']==img]
            display(selection[['titolo', 'url','img']])
            count=count+1
    if count == 0:
        print("Nessuna img corrotta")
    return selection


## Run Changes

In [6]:
sn_list = [luoghi,curiosita,personaggi,cucina,leggende,eccellenze,interviste]


for sn in sn_list:
    nok_url = check_url(sn,sn.url)
    change_url(sn,nok_url)
    selection = check_img(sn)
    if selection is not None:
        change_img(sn,selection)


Nessun Url corrotto
Nessuna img corrotta
Nessuna Img corrotta
Nessun Url corrotto
Nessuna img corrotta
Nessuna Img corrotta
Nessun Url corrotto
Nessuna img corrotta
Nessuna Img corrotta
Nessun Url corrotto
Nessuna img corrotta
Nessuna Img corrotta
Nessun Url corrotto
Nessuna img corrotta
Nessuna Img corrotta
Nessuna img corrotta
Nessuna Img corrotta
Nessun Url corrotto
Nessuna img corrotta
Nessuna Img corrotta


In [ ]:
# Nel caso 
storie_napoli = pd.concat([luoghi,curiosita,personaggi,cucina,leggende,eccellenze,interviste])

Create new geojson

In [3]:
luoghi.to_file("sn_json/luoghi.json")
curiosita.to_file("sn_json/curiosita.json")
personaggi.to_file("sn_json/personaggi.json")
cucina.to_file("sn_json/cucina.json")
eccellenze.to_file("sn_json/eccellenze.json")
leggende.to_file("sn_json/leggende.json")
interviste.to_file("sn_json/interviste.json")

## Ricerca di una determinata stringa nel geodataframe

In [25]:
mask = curiosita['titolo'].str.contains(r"Salerno, la città che fu", na=False)
curiosita.loc[mask]

,type,geometry,titolo,img,author,descr,url,category
5,Feature,POINT (14.75881 40.67999),"Salerno, la città che fu capitale d’Italia per 5 mesi",https://storienapoli.it/wp-content/uploads/2019/05/11-febbraio-Salerno-Capitale.jpg,Federico Quagliuolo,"“Salerno Capitale” è una espressione comune che ricorda uno dei periodi più travagliati della storia d’Italia: per poco tempo il governo italiano si spostò a Salerno, rendendola di fatto capitale…",https://storienapoli.it/2019/05/30/salerno-la-citta-che-fu-capitale-ditalia-per-5-mesi/,curiosita


## Search

In [ ]:
search = requests.get("https://storienapoli.it//wp-json/wp/v2/search/?search=La Méhari di Giancarlo Siani al PAN: eterno simbolo di chi non si arrende")
search_df = pd.DataFrame(search.json())
search.json()[0]['url']

'https://storienapoli.it/2020/09/01/citroen-la-mehari-giancarlo-siani/'

# Get OGR Image

In [15]:
url='https://storienapoli.it/2020/08/19/panicocoli-villaricca-storia-nome/	'
get_ogr_image(url)

'https://storienapoli.it/wp-content/uploads/2022/03/Home-Storie-di-Napoli.jpg'

In [46]:
change_img(luoghi)

In [54]:
selection = check_img(interviste)
if selection is not None:
    change_img(interviste,selection)

Nessun Url corrotto


# 3 Get All information

In [4]:

def get_all(url):
    # Effettua la richiesta HTTP per ottenere il contenuto della pagina
    response = requests.get(url,headers=headers,verify=False)
    html_content = response.text

    # Parsing dell'HTML con BeautifulSoup
    soup = BeautifulSoup(html_content, 'html.parser')

    # Trova il tag meta con attributo property="og:title"
    og_title_tag = soup.find('meta', attrs={'property': 'og:title'})
    title = og_title_tag['content']

    # Trova il tag meta con attributo property="twitter:data1"
    og_author_tag = soup.find('meta', attrs={'name': 'twitter:data1'})
        # Estrai author
    if og_author_tag and 'content' in og_author_tag.attrs:
        author = og_author_tag['content']
        
    else:
        author=''

    # Trova il tag meta con attributo property="og:description"
    og_description_tag = soup.find('meta', attrs={'property': 'og:description'})
        # Estrai l'URL dell'immagine
    if og_description_tag and 'content' in og_description_tag.attrs:
        description = og_description_tag['content']
        
    else:
        description=''



    # Trova il tag meta con attributo property="og:image"
    og_image_tag = soup.find('meta', attrs={'property': 'og:image'})

    # Estrai l'URL dell'immagine
    if og_image_tag and 'content' in og_image_tag.attrs:
        image_url = og_image_tag['content']
        
    else:
        image_url=''
    infos={'title':title,'author':author,'description':description,'image_url':image_url}
    #print(infos['description'])
    return infos




# 4 Get the latest Posts and Insert stories

In [5]:
def get_description(url):
    # Effettua la richiesta HTTP per ottenere il contenuto della pagina
    response = requests.get(url,headers=headers,verify=False)
    html_content = response.text

    # Parsing dell'HTML con BeautifulSoup
    soup = BeautifulSoup(html_content, 'html.parser')

    

    # Trova il tag meta con attributo property="og:description"
    og_description_tag = soup.find('meta', attrs={'property': 'og:description'})
        # Estrai l'URL dell'immagine
    if og_description_tag and 'content' in og_description_tag.attrs:
        description = og_description_tag['content']
        
    else:
        description=''

    #print(infos['description'])
    return description


def train_sn_model(df,titolo,descr):
    # Suddivisione dei dati in training set e test set
    X = df['titolo'] + ' ' + df['descr']  # Concateniamo titolo e descrizione
    y = df['category']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


    # Estrai le feature dai testi utilizzando la rappresentazione TF-IDF (Term Frequency-Inverse Document Frequency)
    vectorizer = TfidfVectorizer()
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)

    # Addestra il modello SVM
    svm = SVC(kernel='linear')  # Puoi sperimentare con diversi kernel
    svm.fit(X_train_tfidf, y_train)

        # Effettua le previsioni
    y_pred = svm.predict(X_test_tfidf)
    new_data = [titolo,descr]
    new_data_tfidf = vectorizer.transform(new_data)
    predicted_label = svm.predict(new_data_tfidf)
    return predicted_label[0]


def insert_story(df,url,category,lon,lat):
    infos=get_all(url)

    df.loc[len(df),['type','geometry','titolo','img','author','descr','url','category']] = ['Feature',Point(lon, lat),infos['title'],infos['image_url'],infos['author'],infos['description'],url,category]
  
def get_tags(tags):
    tags_name=[]
    for tag in tags:
        name = requests.get("https://storienapoli.it/wp-json/wp/v2/tags/"+str(tag),verify=False)
        tags_name.append(name.json()['name'])
    return tags_name



posts = requests.get("https://storienapoli.it/wp-json/wp/v2/posts",verify=False)
posts_df = pd.DataFrame(posts.json())
for index,row in posts_df.iterrows():
    description=get_description(row['link'])

    category = train_sn_model(pd.concat([luoghi,curiosita,personaggi,cucina,leggende,eccellenze,interviste]),row['title']['rendered'],description)
    if category=='luoghi':
        print("insert_story(luoghi,'"+row['link']+"','"+category+"', lon , lat)")
    if category=='curiosita':
        print("insert_story(curiosita,'"+row['link']+"','"+category+"', lon , lat)")
    if category=='personaggi':
        print("insert_story(personaggi,'"+row['link']+"','"+category+"', lon , lat)")
    if category=='leggende':
        print("insert_story(leggende,'"+row['link']+"','"+category+"', lon , lat)")
    if category=='cucina':
        print("insert_story(cucina,'"+row['link']+"','"+category+"', lon , lat)")
    if category=='interviste':
        print("insert_story(interviste,'"+row['link']+"','"+category+"', lon , lat)")
    if category=='eccellenze':
        print("insert_story(eccellenze,'"+row['link']+"','"+category+"', lon , lat)")

insert_story(personaggi,'https://storienapoli.it/2024/08/03/teresa-filangieri-lilluminismo-napoletano-al-femminile/','personaggi', lon , lat)
insert_story(personaggi,'https://storienapoli.it/2024/08/03/gaetano-forte-eroe-sconosciuto/','personaggi', lon , lat)
insert_story(curiosita,'https://storienapoli.it/2024/07/30/le-carte-napoletane-tra-storia-e-tradizione/','curiosita', lon , lat)
insert_story(luoghi,'https://storienapoli.it/2024/07/29/la-chiesa-di-santa-maria-dellanima-fede-e-tradizione-tra-popoli/','luoghi', lon , lat)
insert_story(personaggi,'https://storienapoli.it/2024/07/27/massimo-ranieri-e-perdere-lamore-una-canzone-intramontabile/','personaggi', lon , lat)
insert_story(curiosita,'https://storienapoli.it/2024/07/26/visitando-la-sirena-del-tirreno-cosa-vedere-a-napoli/','curiosita', lon , lat)
insert_story(curiosita,'https://storienapoli.it/2024/07/26/artemisia-gentileschi-il-capolavoro-perduto-torna-a-napoli-dopo-400-anni/','curiosita', lon , lat)
insert_story(luoghi,'http

In [6]:
insert_story(personaggi,'https://storienapoli.it/2024/08/03/teresa-filangieri-lilluminismo-napoletano-al-femminile/','personaggi', 14.2289 , 40.8345)
insert_story(luoghi,'https://storienapoli.it/2024/07/29/la-chiesa-di-santa-maria-dellanima-fede-e-tradizione-tra-popoli/','luoghi', 14.2374 , 40.8387)
insert_story(luoghi,'https://storienapoli.it/2024/07/25/fede-tradizione-e-folklore-la-festa-a-mare-agli-scogli-di-santanna/','luoghi', 13.9591 , 40.7286)
insert_story(luoghi,'https://storienapoli.it/2024/07/24/un-miracolo-di-santantonio-salvo-un-innocente-a-napoli/','luoghi', 14.2582 , 40.8510)
insert_story(personaggi,'https://storienapoli.it/2024/07/23/sepolcro-di-virgilio/','personaggi', 14.2173 , 40.8293)
insert_story(luoghi,'https://storienapoli.it/2024/07/21/la-cappella-di-zi-andrea-gioiello-barocco-di-san-domenico-maggiore/','luoghi',  14.2545 , 40.8488)


In [8]:
import leafmap.foliumap as leafmap
import folium
# Crea una mappa
m = leafmap.Map(location=[41,14], zoom_start=9)  # Imposta la posizione e lo zoom iniziali
m.add_child(folium.LatLngPopup())
m

# 6 Copy and run results

In [3]:
luoghi.to_file("sn_json/luoghi.json")
curiosita.to_file("sn_json/curiosita.json")
personaggi.to_file("sn_json/personaggi.json")
cucina.to_file("sn_json/cucina.json")
eccellenze.to_file("sn_json/eccellenze.json")
leggende.to_file("sn_json/leggende.json")
interviste.to_file("sn_json/interviste.json")


# Get Coordinates by Folium

<hr width=”150px” size=”1″ color=”red” align=”center“ />


Ultima Storia Mappata:https://storienapoli.it/2023/10/14/festa-dei-gigli-di-barra/


Provare ad estrarre le category con uno script di machine learning 

In [3]:
storie_napoli = pd.concat([luoghi,curiosita,personaggi,cucina,leggende,eccellenze,interviste])

In [10]:
len(storie_napoli)

1578

In [27]:
storie_napoli.head(1)

,type,titolo,img,author,descr,url,category,geometry
0,Feature,La Méhari di Giancarlo Siani al PAN: eterno simbolo di chi non si arrende,https://storienapoli.it/wp-content/uploads/2020/09/La-Mèhari.jpg,Roberto Iossa,"Al Palazzo delle Arti di Napoli è esposta la Citroén Méhari di Giancarlo Siani, giornalista ucciso dalla camorra nel 1985",https://storienapoli.it/2020/09/01/citroen-la-mehari-giancarlo-siani/,luoghi,POINT (14.23678 40.83679)


In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

# Carica il tuo DataFrame, che deve contenere due colonne: 'title', 'description', e una colonna 'label' per le etichette.
# Assicurati che le etichette siano codificate in un formato numerico (ad esempio, 0, 1, 2, ...) anziché testo.
df = pd.concat([luoghi,curiosita,personaggi,cucina,leggende,eccellenze,interviste])
# Crea un oggetto LabelEncoder
#label_encoder = LabelEncoder()
#df['label'] = label_encoder.fit_transform(df['category'])

# Suddivisione dei dati in training set e test set
X = df['titolo'] + ' ' + df['descr']  # Concateniamo titolo e descrizione
y = df['category']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Estrai le feature dai testi utilizzando la rappresentazione TF-IDF (Term Frequency-Inverse Document Frequency)
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Addestra il modello SVM
svm = SVC(kernel='linear')  # Puoi sperimentare con diversi kernel
svm.fit(X_train_tfidf, y_train)

    # Effettua le previsioni
y_pred = svm.predict(X_test_tfidf)

# Valuta le previsioni
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.2f}')


Accuracy: 0.62


In [65]:

# Ora puoi utilizzare il modello addestrato per fare previsioni su nuovi dati
titolo="La storia della (dis)avventura di Andreuccio da Perugia"
descr="La storia della (dis)avventura di Andreuccio da Perugia, protagonista di una delle novelle del Decamorone di Boccaccio."
new_data = [titolo,descr]
new_data_tfidf = vectorizer.transform(new_data)
predicted_label = svm.predict(new_data_tfidf)
print(f'Etichetta prevista: {predicted_label[0]}')


Etichetta prevista: luoghi


In [5]:
import subprocess
subprocess.call(["git", "pull"])

128

In [16]:
subprocess.run("git commit -m 'update'")

CompletedProcess(args="git commit -m 'update'", returncode=128)

In [4]:
storie_napoli.to_csv("storie_napoli.csv")